# MOP 3231 · Task 2 — Decision Trees: Entropy, Over-fitting and a Logistic Regression Comparison

**Student:** Imamsidikov · **Dataset:** `data/heart_disease.csv` (Cleveland subset of UCI Heart Disease, 297 patients, CC BY 4.0)

| Part | Topic |
|---|---|
| 0 | Setup, versions, SEED, load and split |
| 1 | Entropy and information gain by hand |
| 2 | A decision tree from scratch (NumPy) and a check against scikit-learn |
| 3 | How deep should the tree be? |
| 4 | Tree or logistic regression? |
| 5 | Reading a tree, and reproducibility |

This is a teaching dataset, not clinical evidence: no medical conclusions are drawn from it.

## Part 0 · Setup

Virtual environment `.Imamsidikov_ML_task_2` (Python 3.12) with the pinned `requirements.txt`; Jupyter is only the tool and stays out of it.

**SEED = 2954**: the last four digits of my student ID (S24072954). It seeds the train/test split and the cross-validation folds.

In [1]:
import sys

import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__)

3.12.12 2.0.2 2.3.3 1.6.1


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 2954

data = pd.read_csv("data/heart_disease.csv")
feature_names = [c for c in data.columns if c != "disease"]
X, y = data[feature_names].to_numpy(), data["disease"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)
print(data.shape, round(y.mean(), 3), X_train.shape, X_test.shape)

(297, 14) 0.461 (207, 13) (90, 13)


## Part 1 · Entropy and information gain by hand

Twelve synthetic patients (6 with disease, 6 without), four yes/no features.
$H = -\sum_c p_c \log_2 p_c$, and $IG = H(\text{parent}) - \sum_k \frac{n_k}{n} H(\text{child}_k)$.

Entropies used below (computed once):
- $H(\tfrac12,\tfrac12) = 1$
- $H(\tfrac56,\tfrac16) = -\tfrac56\log_2\tfrac56 - \tfrac16\log_2\tfrac16 = 0.2192 + 0.4308 = 0.6500$
- $H(\tfrac23,\tfrac13) = -\tfrac23\log_2\tfrac23 - \tfrac13\log_2\tfrac13 = 0.3900 + 0.5283 = 0.9183$
- $H(\tfrac14,\tfrac34) = -\tfrac14\log_2\tfrac14 - \tfrac34\log_2\tfrac34 = 0.5000 + 0.3113 = 0.8113$
- $H(\tfrac25,\tfrac35) = -\tfrac25\log_2\tfrac25 - \tfrac35\log_2\tfrac35 = 0.5288 + 0.4422 = 0.9710$
- $H(\tfrac15,\tfrac45) = -\tfrac15\log_2\tfrac15 - \tfrac45\log_2\tfrac45 = 0.4644 + 0.2575 = 0.7219$
- a pure node has $H = 0$ (using $0\cdot\log_2 0 = 0$)

### 1. Parent entropy

6 yes, 6 no out of 12:
$$H(\text{parent}) = -\tfrac{6}{12}\log_2\tfrac{6}{12} - \tfrac{6}{12}\log_2\tfrac{6}{12} = 0.5 + 0.5 = \mathbf{1.000 \text{ bit}}$$

### 2. Root: gain of each feature (all 12 patients)

| Feature | value = 1 (yes / no) | value = 0 (yes / no) | Weighted child entropy | Gain |
|---|---|---|---|---|
| age_55_plus | p1–p5, p7: 5 / 1 → 0.6500 | p6, p8–p12: 1 / 5 → 0.6500 | $\tfrac{6}{12}0.6500 + \tfrac{6}{12}0.6500 = 0.6500$ | $1 - 0.6500 = \mathbf{0.3500}$ |
| exercise_angina | p1–p4: 4 / 0 → 0 | p5–p12: 2 / 6 → 0.8113 | $\tfrac{4}{12}\cdot 0 + \tfrac{8}{12}0.8113 = 0.5409$ | $1 - 0.5409 = \mathbf{0.4591}$ |
| male | p1–p3, p7–p9: 3 / 3 → 1 | p4–p6, p10–p12: 3 / 3 → 1 | $\tfrac{6}{12}\cdot 1 + \tfrac{6}{12}\cdot 1 = 1$ | $1 - 1 = \mathbf{0.0000}$ |
| high_bp | p1, p2, p4, p5, p10, p11: 4 / 2 → 0.9183 | p3, p6–p9, p12: 2 / 4 → 0.9183 | $\tfrac{6}{12}0.9183 + \tfrac{6}{12}0.9183 = 0.9183$ | $1 - 0.9183 = \mathbf{0.0817}$ |

**Root: `exercise_angina`** (gain 0.4591). Its "yes" branch (p1–p4) is pure: all four have the disease.

### 3. One level down: the impure branch `exercise_angina = 0`

Patients p5–p12: 2 yes (p5, p6), 6 no → $H = H(\tfrac14,\tfrac34) = 0.8113$.

| Feature | value = 1 (yes / no) | value = 0 (yes / no) | Weighted child entropy | Gain |
|---|---|---|---|---|
| age_55_plus | p5, p7: 1 / 1 → 1 | p6, p8–p12: 1 / 5 → 0.6500 | $\tfrac{2}{8}\cdot 1 + \tfrac{6}{8}0.6500 = 0.7375$ | $0.8113 - 0.7375 = \mathbf{0.0738}$ |
| male | p7, p8, p9: 0 / 3 → 0 | p5, p6, p10–p12: 2 / 3 → 0.9710 | $\tfrac{3}{8}\cdot 0 + \tfrac{5}{8}0.9710 = 0.6069$ | $0.8113 - 0.6069 = \mathbf{0.2044}$ |
| high_bp | p5, p10, p11: 1 / 2 → 0.9183 | p6–p9, p12: 1 / 4 → 0.7219 | $\tfrac{3}{8}0.9183 + \tfrac{5}{8}0.7219 = 0.7956$ | $0.8113 - 0.7956 = \mathbf{0.0157}$ |

**`male` is chosen** inside this branch with gain 0.2044, although at the root its gain was exactly 0.

**Why:** at the root, `male` splits the 12 patients into two 3/3 halves, because its effect is cancelled by the
four angina patients (three of whom are male). Once the root has removed those patients, the rest are not mixed the same way:
all three remaining men are healthy, so `male` now separates the classes well. Information gain measures a feature *given
the splits above it*, so a feature that is useless on its own can become the best choice inside a branch.

In [3]:
# The same numbers as arithmetic, to check the hand calculations above
from math import log2

def H(*counts):
    n = sum(counts)
    return -sum(c / n * log2(c / n) for c in counts if c > 0)

parent = H(6, 6)
root_gains = {
    "age_55_plus": parent - (6 / 12 * H(5, 1) + 6 / 12 * H(1, 5)),
    "exercise_angina": parent - (4 / 12 * H(4, 0) + 8 / 12 * H(2, 6)),
    "male": parent - (6 / 12 * H(3, 3) + 6 / 12 * H(3, 3)),
    "high_bp": parent - (6 / 12 * H(4, 2) + 6 / 12 * H(2, 4)),
}
branch = H(2, 6)
branch_gains = {
    "age_55_plus": branch - (2 / 8 * H(1, 1) + 6 / 8 * H(1, 5)),
    "male": branch - (3 / 8 * H(0, 3) + 5 / 8 * H(2, 3)),
    "high_bp": branch - (3 / 8 * H(1, 2) + 5 / 8 * H(1, 4)),
}
print(f"parent entropy = {parent:.4f}")
print("root gains:  ", {k: round(v, 4) for k, v in root_gains.items()})
print(f"branch entropy (exercise_angina = 0) = {branch:.4f}")
print("branch gains:", {k: round(v, 4) for k, v in branch_gains.items()})

parent entropy = 1.0000
root gains:   {'age_55_plus': 0.35, 'exercise_angina': 0.4591, 'male': 0.0, 'high_bp': 0.0817}
branch entropy (exercise_angina = 0) = 0.8113
branch gains: {'age_55_plus': 0.0738, 'male': 0.2044, 'high_bp': 0.0157}


## Part 2 · A decision tree from scratch

The learning algorithm from the lecture in NumPy. Rules followed:
- candidate thresholds are midpoints between consecutive distinct sorted values; rows with `x <= threshold` go left;
- a split is valid only if both children have at least `min_samples_leaf` rows; ties between equal gains go to the lower
  feature index, then the lower threshold (features and thresholds are scanned in increasing order and only a *strictly*
  larger gain replaces the current best);
- a node becomes a leaf when it is pure, when `depth == max_depth`, or when no valid split exists; a leaf predicts its majority
  class, a tie going to the smaller label (`np.argmax` returns the first maximum).

In [4]:
def entropy(y):
    """Entropy in bits of an array of integer class labels."""
    y = np.asarray(y)
    if y.size == 0:
        return 0.0
    p = np.bincount(y) / y.size
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def information_gain(y, left):
    """Entropy of y minus the size-weighted entropy of y[left] and y[~left]; left is a boolean mask."""
    y, left = np.asarray(y), np.asarray(left, dtype=bool)
    n, n_left = y.size, left.sum()
    return entropy(y) - n_left / n * entropy(y[left]) - (n - n_left) / n * entropy(y[~left])


def best_split(X, y, min_samples_leaf=1):
    """Return (feature_index, threshold, gain) of the best split, or None if no valid split exists."""
    best = None
    n = len(y)
    for j in range(X.shape[1]):
        values = np.unique(X[:, j])
        for threshold in (values[:-1] + values[1:]) / 2:
            left = X[:, j] <= threshold
            n_left = left.sum()
            if n_left < min_samples_leaf or n - n_left < min_samples_leaf:
                continue
            gain = information_gain(y, left)
            if best is None or gain > best[2] + 1e-12:  # strictly better; tolerance absorbs float noise
                best = (j, float(threshold), gain)
    return best


def build_tree(X, y, max_depth=None, min_samples_leaf=1, depth=0):
    """Grow a tree recursively and return it as nested dicts (or any structure you like)."""
    counts = np.bincount(y, minlength=2)
    leaf = {"leaf": True, "label": int(np.argmax(counts)), "n": len(y), "counts": counts.tolist()}
    if counts.max() == len(y) or (max_depth is not None and depth == max_depth):
        return leaf
    split = best_split(X, y, min_samples_leaf)
    if split is None:
        return leaf
    j, threshold, gain = split
    left = X[:, j] <= threshold
    return {
        "leaf": False, "feature": j, "threshold": threshold, "gain": gain, "n": len(y),
        "left": build_tree(X[left], y[left], max_depth, min_samples_leaf, depth + 1),
        "right": build_tree(X[~left], y[~left], max_depth, min_samples_leaf, depth + 1),
    }


def predict_tree(tree, X):
    """Return one predicted label per row of X."""
    def predict_row(node, row):
        while not node["leaf"]:
            node = node["left"] if row[node["feature"]] <= node["threshold"] else node["right"]
        return node["label"]
    return np.array([predict_row(tree, row) for row in X])


def n_leaves(tree):
    """Number of leaves of a tree built by build_tree."""
    return 1 if tree["leaf"] else n_leaves(tree["left"]) + n_leaves(tree["right"])

In [5]:
# Sanity check: an unlimited tree is consistent with the training data (no two identical rows with different labels)
tree_full = build_tree(X_train, y_train)
print("leaves:", n_leaves(tree_full), "| training accuracy:", (predict_tree(tree_full, X_train) == y_train).mean())

# min_samples_leaf is respected: no leaf smaller than 10 rows
def leaf_sizes(tree):
    return [tree["n"]] if tree["leaf"] else leaf_sizes(tree["left"]) + leaf_sizes(tree["right"])

print("smallest leaf with min_samples_leaf=10:", min(leaf_sizes(build_tree(X_train, y_train, min_samples_leaf=10))))

leaves: 33 | training accuracy: 1.0
smallest leaf with min_samples_leaf=10: 10
